# Phase 1 — Lexical and Binding Probes (Colab)

Runs Phase 1 of the semantic-flow project on a free Colab T4 GPU.

**Before running:** set runtime to GPU via *Runtime → Change runtime type → T4 GPU*.

**Estimated time:** ~5 min setup, ~1 min extraction, ~2 min probing.

## 1. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → T4 GPU.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive

Results are saved to Drive so they persist after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/semantic-flow/results/phase1'
print(f"Results will be saved to: {DRIVE_OUT}")

## 3. Clone repo and install

In [ ]:
import os

if not os.path.exists('/content/semantic-flow'):
    !git clone https://github.com/virginiaceccatelli/semantic-flow.git /content/semantic-flow

%cd /content/semantic-flow
!pip install -e '.[dev]' -q

## 4. Generate synthetic data

In [ ]:
from src.data.generator import SyntheticCodeGenerator
from src.data.dataset import save_jsonl

gen = SyntheticCodeGenerator(seed=42)
examples = gen.generate_batch(n_binding=100, n_taint=100, n_shadow=50)
save_jsonl(examples, 'data/synthetic/phase1_binding.jsonl')
print(f"Generated {len(examples)} examples")

## 5. Load model

DeepSeek-Coder 1.3B in float16 uses ~2.6 GB VRAM — well within the T4's 15 GB.

In [ ]:
from src.models.loader import ModelConfig, ModelLoader

config = ModelConfig.from_registry('deepseek-coder-1.3b')
print(f"Probing layers: {config.probe_layers}")

loader = ModelLoader(config)
model = loader.model
tokenizer = loader.tokenizer

device = next(model.parameters()).device
used_vram = torch.cuda.memory_allocated() / 1e9
print(f"Model on {device} | VRAM used: {used_vram:.1f} GB")

## 6. Run Phase 1

Extracts hidden states for all examples, then trains and cross-validates:
- **Lexical probe** — token type classification (keyword / identifier / literal / operator)
- **Binding probe** — pairwise: do two token positions refer to the same binding?
- **Binding decoy split** — same-name vs different-name pairs (tests lexical vs semantic tracking)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

from src.data.dataset import CodeProbeDataset
from src.experiments.phase1_lexical import run_phase1
from src.probes.base import ProbeConfig

dataset = CodeProbeDataset.load('data/synthetic/phase1_binding.jsonl')
print(f"Loaded {len(dataset)} examples")

probe_config = ProbeConfig(cv_folds=5, max_iter=500)

results = run_phase1(
    model=model,
    tokenizer=tokenizer,
    examples=dataset.examples,
    layers=config.probe_layers,
    output_dir='results/phase1',
    config=probe_config,
)

print("\nTasks completed:", list(results.keys()))

## 7. Inspect results

In [ ]:
import pandas as pd
from pathlib import Path

result_dir = Path('results/phase1')
for csv in sorted(result_dir.glob('*.csv')):
    df = pd.read_csv(csv)
    print(f"\n--- {csv.stem} ---")
    print(df.to_string(index=False))

## 8. Plot layer curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

result_dir = Path('results/phase1')
dfs = []
for csv in sorted(result_dir.glob('*.csv')):
    df = pd.read_csv(csv)
    df['task'] = csv.stem
    dfs.append(df)

if dfs:
    combined = pd.concat(dfs)
    fig, axes = plt.subplots(1, len(dfs), figsize=(5 * len(dfs), 4), squeeze=False)
    for ax, (task, grp) in zip(axes[0], combined.groupby('task')):
        if 'layer' in grp.columns and 'accuracy' in grp.columns:
            ax.plot(grp['layer'], grp['accuracy'], marker='o', label='accuracy')
            if 'selectivity' in grp.columns:
                ax.plot(grp['layer'], grp['selectivity'], marker='s', linestyle='--', label='selectivity')
            ax.set_title(task)
            ax.set_xlabel('Layer')
            ax.set_ylabel('Score')
            ax.legend()
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/phase1/layer_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Save results to Google Drive

In [ ]:
import shutil

shutil.copytree('results/phase1', DRIVE_OUT, dirs_exist_ok=True)
print(f"Saved to {DRIVE_OUT}")
print("Files:")
for f in sorted(Path(DRIVE_OUT).iterdir()):
    print(f"  {f.name}")